# 05 Clasificación de eventos extremos — ENSO
## Proyecto cambio climático | Dataset: Cambio_climatico.csv (1950–2026)

Este notebook aplica clasificación supervisada para predecir si un mes
registra un **evento climático extremo** (`Evento_Extremo = 1`) a partir de
variables temporales y de fase, sin usar la temperatura ni la anomalía directamente.

# 1. Importación de librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              ConfusionMatrixDisplay)

import warnings
warnings.filterwarnings('ignore')

# 2. Carga del dataset

In [ ]:
df = pd.read_csv('../data/processed/enso_model_ready.csv')
print('Shape:', df.shape)
print('Distribución del target:')
print(df['Evento_Extremo'].value_counts())
print(f'Eventos extremos: {df["Evento_Extremo"].sum()} ({round(df["Evento_Extremo"].mean()*100,2)}%)')
df.head()

# 3. Variable objetivo y features

**Variable objetivo:** `Evento_Extremo` (1 = mes con anomalía extrema, 0 = mes normal)

Se usan solo features **temporales e independientes** — se excluyen deliberadamente
`Anomalia_C`, `Intensidad_Evento` y las temperaturas porque son derivadas del target
o lo predicen trivialmente.

**Features usadas:**
- Numéricas: `Anio`, `Mes`, `Trimestre`, `Duracion_Meses`
- Categóricas: `Decada`, `Fase_Evento`

In [ ]:
TARGET = 'Evento_Extremo'
numeric_features    = ['Anio', 'Mes', 'Trimestre', 'Duracion_Meses']
categorical_features = ['Decada', 'Fase_Evento']

X = df[numeric_features + categorical_features]
y = df[TARGET]

print('Features numéricas:', numeric_features)
print('Features categóricas:', categorical_features)
print('Shape X:', X.shape)
print('Balance clases:', dict(y.value_counts()))

# 4. Preprocesamiento

In [ ]:
num_t = Pipeline([('scaler', StandardScaler())])
cat_t = Pipeline([('ohe', OneHotEncoder(handle_unknown='ignore'))])

preprocessor = ColumnTransformer([
    ('num', num_t, numeric_features),
    ('cat', cat_t, categorical_features)
])

# 5. División train/test

Se usa `stratify=y` para mantener la proporción de eventos extremos en ambos conjuntos.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Train:', X_train.shape, '| Positivos:', y_train.sum())
print('Test: ', X_test.shape,  '| Positivos:', y_test.sum())

# 6. Modelado predictivo — Clasificación

Se comparan cuatro clasificadores con `class_weight='balanced'` para
compensar el desbalance de clases (solo ~9% son eventos extremos).

## 6.1 Regresión Logística

In [ ]:
lr_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(class_weight='balanced', max_iter=500, random_state=42))
])
lr_model.fit(X_train, y_train)
lr_preds = lr_model.predict(X_test)
lr_proba = lr_model.predict_proba(X_test)[:,1]

## 6.2 Árbol de Decisión

In [ ]:
tree_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=42))
])
tree_model.fit(X_train, y_train)
tree_preds = tree_model.predict(X_test)
tree_proba = tree_model.predict_proba(X_test)[:,1]

## 6.3 Random Forest

In [ ]:
rf_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42))
])
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)[:,1]

## 6.4 Gradient Boosting

In [ ]:
gb_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', GradientBoostingClassifier(random_state=42))
])
gb_model.fit(X_train, y_train)
gb_preds = gb_model.predict(X_test)
gb_proba = gb_model.predict_proba(X_test)[:,1]

# 7. Evaluación de modelos

In [ ]:
def evaluate(y_true, y_pred, y_prob, name):
    return {
        'Modelo':    name,
        'Accuracy':  round(accuracy_score(y_true, y_pred), 4),
        'Precision': round(precision_score(y_true, y_pred, zero_division=0), 4),
        'Recall':    round(recall_score(y_true, y_pred, zero_division=0), 4),
        'F1':        round(f1_score(y_true, y_pred, zero_division=0), 4),
        'AUC-ROC':   round(roc_auc_score(y_true, y_prob), 4),
    }

results = [
    evaluate(y_test, lr_preds,   lr_proba,   'Regresión Logística'),
    evaluate(y_test, tree_preds, tree_proba, 'Árbol de Decisión'),
    evaluate(y_test, rf_preds,   rf_proba,   'Random Forest'),
    evaluate(y_test, gb_preds,   gb_proba,   'Gradient Boosting'),
]
results_df = pd.DataFrame(results)
results_df

# 8. Validación cruzada (Random Forest)

In [ ]:
cv_scores = cross_val_score(rf_model, X, y, cv=5, scoring='f1')
print('Scores CV F1:', np.round(cv_scores, 4))
print(f'Promedio F1 CV : {round(cv_scores.mean(), 4)}')
print(f'Desviación std : {round(cv_scores.std(), 4)}')

# 9. Comparación visual de modelos

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

metrics = ['F1','AUC-ROC','Recall','Precision']
results_df.set_index('Modelo')[metrics].plot(kind='bar', ax=axes[0], colormap='Set2')
axes[0].set_title('Métricas de clasificación por modelo')
axes[0].set_ylim(0, 1.1)
axes[0].tick_params(axis='x', rotation=20)
axes[0].legend(loc='upper right', fontsize=8)

# Matriz de confusión — mejor modelo (RF)
cm = confusion_matrix(y_test, rf_preds)
disp = ConfusionMatrixDisplay(cm, display_labels=['Normal','Extremo'])
disp.plot(ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title('Matriz de confusión — Random Forest')

plt.tight_layout()
plt.savefig('../reports/clasificacion_evento_extremo.png', dpi=120, bbox_inches='tight')
plt.show()

# 10. Importancia de variables (Random Forest)

In [ ]:
ohe_cols   = (rf_model.named_steps['preprocessor']
                      .named_transformers_['cat']
                      .named_steps['ohe']
                      .get_feature_names_out(categorical_features))
all_feats  = numeric_features + list(ohe_cols)
importances = rf_model.named_steps['model'].feature_importances_

feat_imp = (pd.DataFrame({'Variable': all_feats, 'Importancia': importances})
              .sort_values('Importancia', ascending=False).head(8))

plt.figure(figsize=(8, 5))
sns.barplot(data=feat_imp, x='Importancia', y='Variable', palette='Greens_d')
plt.title('Top 8 variables — Random Forest')
plt.tight_layout()
plt.savefig('../reports/importancia_clasificacion.png', dpi=120, bbox_inches='tight')
plt.show()
feat_imp

# 11. Interpretación de resultados

## Evaluación de modelos

| Modelo | Accuracy | Precision | Recall | F1 | AUC-ROC |
|---|---|---|---|---|---|
| Regresión Logística | 0.7158 | 0.2188 | 0.8750 | 0.3500 | 0.8312 |
| Árbol de Decisión | 0.8415 | 0.3415 | 0.8750 | 0.4912 | 0.8808 |
| **Random Forest** | **0.9508** | **0.8182** | **0.5625** | **0.6667** | **0.9308** |
| Gradient Boosting | 0.9399 | 0.6667 | 0.6250 | 0.6452 | 0.9203 |

1. **Random Forest** es el mejor modelo con F1=0.6667 y AUC-ROC=0.9308.
2. La **Regresión Logística** tiene el mayor Recall (0.875) pero baja Precision (0.22),
   lo que genera muchas falsas alarmas.
3. El **AUC-ROC** de todos los modelos supera 0.83, indicando buena capacidad
   discriminativa a pesar del desbalance de clases.

## Importancia de variables

| Variable | Importancia |
|---|---|
| `Duracion_Meses` | 20.1% |
| `Fase_Evento_Neutral` | 19.3% |
| `Mes` | 18.9% |
| `Anio` | 13.3% |
| `Trimestre` | 10.2% |

La **duración del evento** y la **fase ENSO** son los predictores dominantes,
confirmando que los eventos extremos tienden a ocurrir dentro de episodios
prolongados de El Niño o La Niña.

# 12. Conclusiones

## División de los datos

- **731 registros** para entrenamiento (80%) | **183 para prueba** (20%)
- Clases balanceadas con `stratify=y`: 16 positivos en test (8.7%)

## Hallazgos principales

1. **Random Forest** logra el mejor balance F1/AUC (0.667 / 0.931) con solo
   features temporales y de fase — sin usar temperatura ni anomalía.
2. La **duración del evento** activo es el predictor más importante (20.1%),
   lo que implica que los eventos extremos casi nunca ocurren en episodios cortos.
3. El **mes del año** influye significativamente (18.9%), reflejando la estacionalidad
   del sistema ENSO (picos típicos en diciembre-febrero).
4. La **validación cruzada** F1 promedio de 0.386 ± 0.138 indica variabilidad entre
   folds, consecuencia del bajo número de eventos positivos (82 de 914 registros).
5. Para aplicaciones de alerta temprana en Colombia, se recomienda priorizar **Recall**
   (detectar todos los eventos extremos aunque se generen falsas alarmas) usando
   el umbral de decisión de la Regresión Logística o ajustando el threshold de RF.